In [1]:
import numpy as np

from enterprise_warp import enterprise_warp

from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc
# ─── Monkey-patch EntryPoint.default_kwargs ────────────────────────────────────
try:
    import entrypoints
    entrypoints.EntryPoint.default_kwargs = property(
        lambda self: getattr(self.load(), "__kwdefaults__", {}) or {}
    )
except ImportError:
    pass

try:
    # In case enterprise_warp is using importlib.metadata.EntryPoint instead
    import importlib.metadata as _im
    _im.EntryPoint.default_kwargs = property(
        lambda self: getattr(self.load(), "__kwdefaults__", {}) or {}
    )
except (ImportError, AttributeError):
    pass
# ────────────────────────────────────────────────────────────────────────────────

MPI startup(): FI_PSM3_UUID was not generated, please set it to avoid possible resources ownership conflicts between MPI processes
Optional acor package is not installed. Acor is optionally used to calculate the effective chain length for output in the chain file.


/home/ezahraou/.local/lib/python3.10/site-packages/enterprise_warp-0.0.2-py3.10.egg/enterprise_warp/results.py:32: UserWarning: ChainConsumer is not available


In [4]:
import sys
import ppta_dr2_models


#"gwb": "pol_dist_10_nfreqs"
# keep only the script name
sys.argv = sys.argv[:1]

path_par = "/fred/oz103/ezahraoui/PPTA/run_wrap/sampling_params/ppta_pol_test_5psr_byband_curn.dat"
#path_par = "/fred/oz103/ezahraoui/PPTA/run_wrap/sampling_params/ppta_dr2_vanilla_curn_full_param.dat"
#opts = enterprise_warp.parse_commandline()
opts = enterprise_warp.parse_commandline()

custom = ppta_dr2_models.PPTADR2Models



In [5]:
#"pol_dist_5_nfreqs" : 
params = enterprise_warp.Params(path_par,opts=opts,custom_models_obj=custom)


------------------
Setting default parameters with file  /fred/oz103/ezahraoui/PPTA/run_wrap/sampling_params/ppta_pol_test_5psr_byband_curn.dat
Setting default Solar System Ephemeris: DE438
Only using pulsars from psrlist
Setting a default linear timing model
Setting timing model SVD to 0 (False)
Including transient events to specific pulsar models
Setting reference radio frequency to 1400 MHz
------------------
Setting sampler kwargs from the parameter file:
------------------
Number of .par files:  25
Number of .tim files:  25
Loading pulsars
[tempo2Util.C:396] Warning: [TIM1] Please place MODE flags in the parameter file 
[tempo2Util.C:401] Warning: [DUP1] duplicated warnings have been suppressed.


Pol calibration is set to True
Adding polynomial distortion vec to pulsars


In [6]:
pta = enterprise_warp.init_pta(params)
print('Pulsar Timing Array: ', len(pta))


Number of Fourier frequencies for the GWB/CPL signal:  30
Using noise parameters from the file:  {'J0711-6830_CASPSR_40CM_efac': 1.1298258736280822, 'J0711-6830_CASPSR_40CM_log10_equad': -7.6299166849010085, 'J0711-6830_CPSR2_20CM_efac': 1.064084985404349, 'J0711-6830_CPSR2_20CM_log10_equad': -6.04717097779411, 'J0711-6830_CPSR2_50CM_efac': 1.0940590289422412, 'J0711-6830_CPSR2_50CM_log10_equad': -7.731224036363758, 'J0711-6830_PDFB1_10CM_efac': 0.9877037281298511, 'J0711-6830_PDFB1_10CM_log10_equad': -5.21005052309488, 'J0711-6830_PDFB1_1433_efac': 1.092925238127751, 'J0711-6830_PDFB1_1433_log10_equad': -7.755054181029302, 'J0711-6830_PDFB1_20CM_efac': 1.082837868905749, 'J0711-6830_PDFB1_20CM_log10_equad': -8.07792126238142, 'J0711-6830_PDFB1_early_10CM_efac': 0.9638568848852437, 'J0711-6830_PDFB1_early_10CM_log10_equad': -5.280023683036845, 'J0711-6830_PDFB1_early_20CM_efac': 0.9999217999305334, 'J0711-6830_PDFB1_early_20CM_log10_equad': -7.707442645769634, 'J0711-6830_PDFB_10CM_efa

In [ ]:
#super_model = hypermodel.HyperModel(pta)

x0 = np.hstack([p.sample() for p in pta[0].params])
ndim = len(x0)
print('ndim: ', ndim)
cov = np.diag(np.ones(ndim) *1**2)
print('Super model parameters: ', pta[0].params)
#sampler = super_model.setup_sampler(resume=True, outdir=params.output_dir)
sampler = ptmcmc(ndim, pta[0].get_lnlikelihood, pta[0].get_lnprior, cov, 
                outDir=params.output_dir, resume=False)
N = int(2e6)

# Remove extra kwargs that Bilby took from PTSampler module, not ".sample"
#ptmcmc_sample_kwargs = inspect.getargspec(sampler.sample).args
# upd_sample_kwargs = {key: val for key, val in params.sampler_kwargs.items()
#                               if key in ptmcmc_sample_kwargs}
#del upd_sample_kwargs['Niter']
#del upd_sample_kwargs['p0']

#sampler.sample(x0, N, **upd_sample_kwargs)
print('len(x0): ', len(x0))


In [13]:
x0 = np.hstack([p.sample() for p in pta[0].params])
x0

array([  9.02356843,  -8.2016756 ,   6.7280153 ,  -5.08377993,
         4.06975296,  -7.53597514,   4.72079911,  -8.1874717 ,
         5.1792236 , -19.0151632 ,   3.70737228,  -6.12568384,
        -8.94316054,   9.41124409,  -7.43659382,  -5.25421423,
         2.86730229,  -6.01962924,  -8.99457182,   0.1629368 ,
       -15.42008081,   7.9863009 , -13.55566698,   2.13037531,
        -8.99623572,  -7.83588521,   4.39429903,  -9.10771295,
        -6.81308552,   4.89712865,  -9.14915371,  -6.16251456,
         3.27812471,  -9.80426417,   0.21791009,  -6.45441561,
         6.77112468,  -7.65007539,  -6.99324354,   8.74803838,
        -5.50547684,  -9.96866433,   3.36047031,  -8.79335861,
        -8.06647478,   7.59460221, -18.19546685,   8.6907507 ,
        -8.11620001,   8.11017587,  -8.14860958,   7.14867596,
        -7.78826432,   3.47599009,  -7.35946498,   5.75518674,
       -12.21490236,   7.5455441 , -15.95700829,   0.87520059,
        -6.54778239,   8.04892929,  -5.8260659 ,   3.71

In [12]:
pta[0].get_lnprior(x0)

-162.93801465418125

In [14]:
pta[0].get_lnlikelihood(x0)

KernelMatrix(134209.3487433)

In [ ]:
sampler.sample(x0, N, SCAMweight=30, AMweight=15,thin=20,)